# MCP and Modern Agentic Systems




## 1. Demo Agent vs Production Agent

A demo agent can be as small as one prompt and one model call. A production agent needs more structure because it may touch real systems.

```text
Demo:
  user -> prompt -> LLM -> answer

Production:
  user -> workflow -> retrieval/tools/memory -> validation -> answer/action -> logs/eval
```

A production design tries to answer four practical questions:

1. What information does the agent need?
2. What actions is the agent allowed to perform?
3. Which actions need validation or human approval?
4. How will we know whether the agent worked correctly?


In [ ]:
import json

demo_agent = {
    "input": "one user message",
    "process": ["send prompt to model"],
    "output": "text answer",
    "main_risk": "no control over multi-step behavior",
}

production_agent = {
    "input": "user goal",
    "process": [
        "classify task",
        "retrieve context",
        "choose tool if needed",
        "validate action",
        "ask approval for risky actions",
        "return final answer",
        "log trace for evaluation",
    ],
    "output": "answer or controlled action",
    "main_risk": "system design complexity",
}

print(json.dumps({"demo": demo_agent, "production": production_agent}, indent=2))


## 2. What Is MCP?

**MCP** means **Model Context Protocol**.

MCP is a standard way for an AI host application to connect to external tools and context through reusable servers.

```text
Host app <-> MCP client <-> MCP server <-> external capability
```

The capability could be a file system, database, documentation search tool, browser, ticketing system, or internal API.

Important point: MCP does not replace the agent workflow. It only standardizes how capabilities are exposed and called.


In [ ]:
from dataclasses import dataclass, asdict
from typing import Any

@dataclass
class MCPToolDescription:
    name: str
    description: str
    input_schema: dict[str, Any]

# A tool description should be specific enough for the model to know when to use it.
search_docs_tool = MCPToolDescription(
    name="search_docs",
    description="Search course documentation and return the most relevant passages.",
    input_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query written in natural language"}
        },
        "required": ["query"],
    },
)

print(json.dumps(asdict(search_docs_tool), indent=2))


## 3. MCP Request Flow

The model usually does not call the external system directly. The host controls the available tools and routes tool calls through the MCP client.

```mermaid
sequenceDiagram
    participant User
    participant Host
    participant Model
    participant Client as MCP Client
    participant Server as MCP Server
    User->>Host: Ask for a task
    Host->>Model: Send prompt and available tools
    Model-->>Host: Request tool call
    Host->>Client: Validate and route request
    Client->>Server: Structured MCP call
    Server-->>Client: Structured result
    Client-->>Host: Return result
    Host->>Model: Add result to context
    Model-->>Host: Final answer
    Host-->>User: Response
```

The host is important because it can enforce permissions, show approval prompts, hide tools that should not be available, and log what happened.


## 4. MCP vs LangGraph vs RAG vs Plain Tool Calling

These concepts can work together, but they are not the same thing.

| Concept | Main job | Example use |
|---|---|---|
| Plain tool calling | Call local functions from one app | `calculate_tax(amount)` |
| MCP | Standardize access to reusable capabilities | shared docs/database/file server |
| LangGraph | Control workflow and state | route -> retrieve -> tool -> approval -> answer |
| RAG | Retrieve relevant knowledge | search policy docs before answering |
| Guardrails | Validate behavior and output | block unsafe action or invalid JSON |


In [ ]:
layers = {
    "workflow_layer": {
        "example": "LangGraph",
        "job": ["routing", "state", "retries", "human interrupts"],
    },
    "capability_layer": {
        "example": "MCP",
        "job": ["files", "database", "docs", "browser", "external APIs"],
    },
    "knowledge_layer": {
        "example": "RAG",
        "job": ["retrieve relevant documents", "ground answers"],
    },
    "safety_layer": {
        "example": "guardrails",
        "job": ["schema validation", "policy checks", "approval rules"],
    },
}

print(json.dumps(layers, indent=2))


## 5. Production Agent Building Blocks

A strong agent architecture separates responsibilities.

```mermaid
flowchart LR
    U[User goal] --> W[Workflow]
    W --> R[Retrieval]
    W --> T[Tools / MCP]
    W --> M[Memory]
    W --> LLM[LLM]
    LLM --> V[Validation]
    V --> A{Risky action?}
    A -- No --> O[Answer / action]
    A -- Yes --> H[Human approval]
    H --> O
    O --> E[Logs and evaluation]
```

The model reasons. The system controls the process.


In [ ]:
building_blocks = [
    ("workflow", "Controls order of steps, routing, retries, and interruptions."),
    ("tools", "Let the agent act on real systems."),
    ("memory", "Stores useful context across a task or across sessions."),
    ("structured_output", "Makes model responses machine-readable."),
    ("guardrails", "Validate actions, inputs, outputs, and policy."),
    ("human_approval", "Pauses risky or irreversible actions."),
    ("observability", "Records traces, tool calls, and errors."),
    ("evaluation", "Measures quality on realistic test cases."),
]

for name, explanation in building_blocks:
    print(f"{name:18} -> {explanation}")


## 6. Memory: What Should Be Remembered?

Memory is useful, but dangerous if used carelessly.

Good memory examples:

- stable preferences
- project conventions
- durable decisions
- repeated user context

Bad memory examples:

- secrets
- temporary errors
- unverified assumptions
- noisy one-time details


In [ ]:
candidate_facts = [
    {"fact": "User prefers notebook outputs cleared before sharing", "stable": True, "safe": True},
    {"fact": "API key is sk-live-123", "stable": True, "safe": False},
    {"fact": "A command failed once because Wi-Fi dropped", "stable": False, "safe": True},
    {"fact": "Project uses Mermaid diagrams in markdown notes", "stable": True, "safe": True},
]

for item in candidate_facts:
    should_store = item["stable"] and item["safe"]
    decision = "store" if should_store else "do not store"
    print(f"{decision:12} -> {item['fact']}")


## 7. Human Approval and Risk

Agents can prepare actions, but they should not automatically perform every action.

Require approval for actions that are costly, irreversible, external-facing, or sensitive.


In [ ]:
actions = [
    {"action": "summarize public documentation", "risk": "low"},
    {"action": "create a draft email", "risk": "medium"},
    {"action": "send the email to a customer", "risk": "high"},
    {"action": "delete production database rows", "risk": "high"},
    {"action": "format a local markdown file", "risk": "low"},
]

for item in actions:
    needs_approval = item["risk"] == "high"
    print(f"{item['action']:<45} approval_required={needs_approval}")


## 8. Example Architecture: Support Assistant

Suppose we are building a support assistant for order questions.

```mermaid
flowchart TD
    Q[Customer question] --> R[Route intent]
    R --> D{Needs private order data?}
    D -- No --> DOCS[Search help docs]
    D -- Yes --> AUTH[Check identity]
    AUTH --> DB[Query order database]
    DOCS --> DRAFT[Draft answer]
    DB --> DRAFT
    DRAFT --> CHECK[Validate policy and tone]
    CHECK --> OUT[Reply]

    REFUND[Refund request] --> POLICY[Check refund policy]
    POLICY --> APPROVAL{Needs approval?}
    APPROVAL -- Yes --> HUMAN[Human review]
    APPROVAL -- No --> ISSUE[Issue refund]
```

In this design, documentation search may be RAG, order lookup may be an MCP database server, and refund execution should usually require approval.


In [ ]:
support_agent_design = {
    "tools": ["search_help_docs", "query_order", "create_ticket", "draft_email"],
    "mcp_candidates": ["search_help_docs", "query_order"],
    "approval_required": ["send_email", "issue_refund", "change_shipping_address"],
    "memory": ["preferred language", "stable account preferences"],
    "eval_cases": [
        "simple FAQ question",
        "order status with identity check",
        "refund request that needs approval",
        "ambiguous complaint requiring clarification",
    ],
}

print(json.dumps(support_agent_design, indent=2))


## 9. Minimal MCP Server Example

The next cell writes a tiny MCP server file. The file exposes two tools:

- `add(a, b)`
- `word_count(text)`

This cell only writes the server code. To run it as a real MCP server, the environment needs the MCP Python SDK installed.


In [ ]:
%%writefile mcp_demo_server.py
from mcp.server.fastmcp import FastMCP

# The server name is shown by the host application when the server is registered.
mcp = FastMCP("demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b

@mcp.tool()
def word_count(text: str) -> int:
    """Count the number of whitespace-separated words in text."""
    return len(text.split())

if __name__ == "__main__":
    # By default this runs over stdio, which many MCP hosts can connect to.
    mcp.run()


## 10. Test the Tool Logic Without Running MCP

The real MCP server wraps ordinary Python functions. Before connecting a host, it is useful to test the underlying logic directly.


In [ ]:
# These are plain Python versions of the same tool behavior.
# This quick check does not require the MCP SDK.

def add(a: int, b: int) -> int:
    return a + b

def word_count(text: str) -> int:
    return len(text.split())

print("add(21, 21) ->", add(21, 21))
print('word_count("MCP makes tools reusable") ->', word_count("MCP makes tools reusable"))


## 11. Registering the Server With a Host

If using Claude Code, the registration pattern is:

```bash
claude mcp add demo -- python mcp_demo_server.py
claude mcp list
```

Then you can ask the host:

```text
Use the demo add tool to add 21 and 21.
Use word_count to count the words in "MCP makes tools reusable".
```

Other hosts use the same idea: configure a command that starts the MCP server, then the host discovers the tools.


## 12. When to Use MCP

Use MCP when:

- tools need to be reused across multiple AI hosts
- permissions and boundaries matter
- the integration has its own lifecycle
- many tools/resources need a standard interface
- you want cleaner separation between the host app and external capability

Skip MCP when:

- the project is a tiny prototype
- one local function is enough
- no other host needs the capability
- MCP setup adds more complexity than value


In [ ]:
use_cases = [
    {"case": "one notebook calls a local helper function", "use_mcp": False},
    {"case": "shared company docs search used by IDE and chat app", "use_mcp": True},
    {"case": "internal database access shared by many agent hosts", "use_mcp": True},
    {"case": "small CSV summarizer script", "use_mcp": False},
]

for item in use_cases:
    recommendation = "MCP is useful" if item["use_mcp"] else "plain function/tool is enough"
    print(f"{item['case']:<65} -> {recommendation}")


## 13. Evaluation Checklist

A production agent should be tested on realistic cases, not only easy demos.


In [ ]:
evaluation_checklist = [
    "Does the agent choose the correct tool?",
    "Does it avoid tools when no tool is needed?",
    "Does it ask for approval before risky actions?",
    "Does it produce valid structured output?",
    "Does it handle tool errors gracefully?",
    "Does it avoid using stale or unsafe memory?",
    "Can we inspect logs/traces after failure?",
]

for i, question in enumerate(evaluation_checklist, start=1):
    print(f"{i}. {question}")
